# 5A · Variance Decomposition & Attribution
### Financial Analytics — Module 5

"Revenue grew 14%." The room nods. Someone asks: **"why?"** — and everything you build in this notebook is the professional answer to that question.

The core move is always the same: **take one difference and split it into named, quantified parts that add back up exactly.** Three versions of the move:

1. **Price × Volume** — did we sell more, or sell dearer?
2. **The cost bridge** — which line drove the profit change?
3. **Brinson attribution** — did the portfolio win by *picking sectors* or *picking stocks*?

Same skeleton, three rooms of finance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
fin = pd.read_csv(BASE + "company_financials.csv")
uni = pd.read_csv(BASE + "nse_stock_universe.csv", parse_dates=["date"])
GREEN, RED, BLUE = "#16A34A", "#DC2626", "#2563EB"
fin[["fiscal_year","revenue_cr","stores_count","avg_ticket_size_inr"]].tail(3)

---
## 1. Price × Volume: did we sell more, or sell dearer?

Revenue = number of transactions × average ticket size. We have revenue and ticket size, so **implied transactions = revenue / ticket** — deriving the missing driver from an identity is itself a diagnostic skill.

In [ ]:
fin["txns_cr"] = fin["revenue_cr"] * 1e7 / fin["avg_ticket_size_inr"] / 1e7   # transactions, in crore

y0 = fin[fin["fiscal_year"] == "FY24-25"].iloc[0]
y1 = fin[fin["fiscal_year"] == "FY25-26"].iloc[0]

d_rev = y1["revenue_cr"] - y0["revenue_cr"]
print(f"Revenue: {y0['revenue_cr']:,.0f} -> {y1['revenue_cr']:,.0f}  (Delta = {d_rev:+,.0f} cr, {d_rev/y0['revenue_cr']*100:+.1f}%)")

In [ ]:
# The decomposition. Rev = Volume x Price, so the change splits into three exact pieces:
vol0, vol1 = y0["txns_cr"], y1["txns_cr"]
p0,   p1   = y0["avg_ticket_size_inr"], y1["avg_ticket_size_inr"]

volume_effect      = (vol1 - vol0) * p0          # more transactions, at OLD prices
price_effect       = (p1 - p0) * vol0            # higher ticket, on OLD volume
interaction_effect = (vol1 - vol0) * (p1 - p0)   # the corner piece: growth x inflation together

print(f"Volume effect      : {volume_effect:+10,.0f} cr   (sold MORE)")
print(f"Price effect       : {price_effect:+10,.0f} cr   (sold DEARER)")
print(f"Interaction        : {interaction_effect:+10,.0f} cr")
print(f"{'-'*46}")
print(f"Sum                : {volume_effect+price_effect+interaction_effect:+10,.0f} cr")
print(f"Actual revenue chg : {d_rev:+10,.0f} cr   -> reconciles: {abs(volume_effect+price_effect+interaction_effect - d_rev) < 0.5}")

**The reconciliation line is the whole discipline.** A decomposition that doesn't add back to the actual change has a leak — and a leak means a wrong story. (Small print for honesty: the *interaction* term is real and belongs to neither pure effect. Many firms fold it into price or split it pro-rata; whichever convention you pick, **state it**. Hidden conventions are how two analysts "reconcile" to different answers.)

In [ ]:
# Draw it as a variance waterfall - Module 4's chart, now carrying a WHY
steps = [("FY24-25\nrevenue", y0["revenue_cr"], BLUE),
         ("Volume", volume_effect, GREEN if volume_effect>0 else RED),
         ("Price", price_effect, GREEN if price_effect>0 else RED),
         ("Interaction", interaction_effect, "#94A3B8"),
         ("FY25-26\nrevenue", y1["revenue_cr"], BLUE)]

fig, ax = plt.subplots(figsize=(9, 4.2))
running = 0
for i, (lab, val, clr) in enumerate(steps):
    if i in (0, len(steps)-1):
        ax.bar(i, val, color=clr); running = val if i == 0 else running
    else:
        ax.bar(i, val, bottom=running, color=clr); running += val
    ax.text(i, (running if 0 < i < len(steps)-1 else val) + 120, f"{val:,.0f}", ha="center", fontsize=9)
ax.set_xticks(range(len(steps)), [s[0] for s in steps])
ax.set_title("Why revenue grew: mostly volume, helped by price", loc="left", fontweight="bold")
ax.set_ylabel("Rs crore"); plt.tight_layout(); plt.show()

### ✏️ Exercise 1 — one more layer of WHY
Volume grew — but was that **more stores** or **more sales per store**? Same identity trick: per-store volume = txns / stores. Decompose the volume effect into a *stores effect* and a *same-store effect* (use the same old-price/old-quantity pattern). Which engine drives MoneyMart?

In [ ]:
# your code here


---
## 2. The cost bridge: which line drove the profit change?

Same move, applied to EBITDA. Every cost line's change is a step; the bridge must land on the actual EBITDA change.

In [ ]:
cost_lines = ["cogs_cr", "employee_cost_cr", "marketing_cr", "other_opex_cr"]

d_ebitda = y1["ebitda_cr"] - y0["ebitda_cr"]
steps = [("Revenue", d_rev)] + [(c.replace("_cr","").replace("_"," ").title(), -(y1[c]-y0[c])) for c in cost_lines]

print(f"EBITDA change to explain: {d_ebitda:+,.1f} cr\n")
total = 0
for lab, val in steps:
    total += val
    print(f"  {lab:<14} {val:+9,.1f} cr")
print(f"  {'-'*26}\n  Explained     {total:+9,.1f} cr   -> reconciles: {abs(total - d_ebitda) < 0.5}")

Read the story in the numbers: revenue added, every cost line took its bite, and the *net* is the EBITDA change. Notice costs are shown with flipped sign (a cost **increase** is a **negative** step for profit) — sign conventions are where variance walks silently break, so fix yours and reconcile.

### ✏️ Exercise 2 — quantify operating leverage (the Module 4 cliffhanger)
For the COVID year (FY19-20 → FY20-21): compute the % change in revenue and the % change in each cost line. The ratio (cost %Δ ÷ revenue %Δ) is that cost's **flexibility** — near 1 = fully variable, near 0 = stubbornly fixed. Rank the four lines. The stubborn ones are *why* profit fell so much harder than revenue: that's operating leverage, quantified.

In [ ]:
# your code here


---
## 3. Brinson attribution: skill at picking sectors, or picking stocks?

The same decomposition instinct, in the asset-management room. Your fund beat (or trailed) its benchmark — was that **allocation** (overweighting the right sectors) or **selection** (picking the right stocks inside each sector)?

Toy setup: a 2-sector world (Financials, IT) built from your stock universe, 2025 returns.

In [ ]:
u25 = uni[uni["date"].dt.year == 2025]
first = u25.sort_values("date").groupby("ticker")["close"].first()
last  = u25.sort_values("date").groupby("ticker")["close"].last()
ret = (last / first - 1).rename("ret").reset_index()
ret = ret.merge(uni[["ticker","sector"]].drop_duplicates(), on="ticker")
ret = ret[ret["sector"].isin(["Financials", "IT"])]

# Benchmark: equal-weight everything. Portfolio: overweight IT + hold only the best-known names.
bench = ret.copy();  bench["w"] = 1/len(bench)
port  = ret[ret["ticker"].isin(["TCS.NS","INFY.NS","HDFCBANK.NS","ICICIBANK.NS"])].copy()
port["w"] = np.where(port["sector"]=="IT", 0.35, 0.15)          # 70% IT, 30% Financials

def sector_summary(df):
    g = df.groupby("sector").apply(lambda x: pd.Series({
        "weight": x["w"].sum(), "ret": np.average(x["ret"], weights=x["w"])}), include_groups=False)
    return g

B, P = sector_summary(bench), sector_summary(port)
summary = B.join(P, lsuffix="_bench", rsuffix="_port")
print(summary.round(3))

total_b = (B["weight"]*B["ret"]).sum(); total_p = (P["weight"]*P["ret"]).sum()
print(f"\nBenchmark return: {total_b:+.2%} | Portfolio return: {total_p:+.2%} | Active: {total_p-total_b:+.2%}")

In [ ]:
# The Brinson split, sector by sector:
#   allocation = (w_port - w_bench) x (bench sector return - bench TOTAL return)
#   selection  =  w_port x (port sector return - bench sector return)
rows = []
for s in summary.index:
    alloc = (summary.loc[s,"weight_port"] - summary.loc[s,"weight_bench"]) * (summary.loc[s,"ret_bench"] - total_b)
    select =  summary.loc[s,"weight_port"] * (summary.loc[s,"ret_port"] - summary.loc[s,"ret_bench"])
    rows.append((s, alloc, select))
attr = pd.DataFrame(rows, columns=["sector","allocation","selection"]).set_index("sector")
print(attr.round(4))
print(f"\nAllocation total : {attr['allocation'].sum():+.2%}   (did overweighting IT help?)")
print(f"Selection total  : {attr['selection'].sum():+.2%}   (did our stock picks beat their sectors?)")
print(f"Sum              : {attr.values.sum():+.2%}  vs active {total_p-total_b:+.2%}  -> reconciles (to rounding)")

One skill, three rooms: **name the parts, quantify each, make them add back up.** In FP&A it's price/volume; on a fund desk it's allocation/selection; in Module 8 the same instinct prices a company. The maths changes costume; the move never does.

### ✏️ Exercise 3
Flip the portfolio: 70% Financials / 30% IT, same four stocks. Recompute. Does allocation flip sign? Does selection? (It shouldn't move much — and *why it shouldn't* is the point: selection measures picks **within** sectors, independent of the sector bet.)

---
*AI disclosure: ______*

In [ ]:
# workspace
